<a href="https://colab.research.google.com/github/Areej973/LangChain---Overview/blob/main/LangChain_Overview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1> LangChain - Overview</h1>

In ths lab, you will learn how get your ways around LangChain. Some of the topics we will cover include:

1. Load and chat with LLMs
2. Parse outputs
3. Chat with external documents
4. Keep conversation history (`memory`)
5. Chaining conversations
6. Working with agent


In [1]:
!pip install -U langchain langchain-community langchain-openai langchainhub

In [2]:
!pip install  \
chromadb faiss-gpu-cu12 \
huggingface_hub transformers sentence-transformers \
ipywidgets tiktoken google-search-results

print("\n All packages installed. Please restart the runtime session to apply the changes.")

  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached faiss_gpu_cu12-1.14.1.post1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pybase64-1.4.3-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (8.7 kB)
  Using cached onnxruntime-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.3 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.42.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-manylinux_2_34_x86_64.whl.metadata (10 kB)
  Using cached kubernetes-36.0.1-py2.py3-none-any.whl.metadata (1.8 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached jedi-0.20.0-py2.py3-none-any.whl.metadata (23 kB)
  U

In [3]:
import os
from langchain_openai import ChatOpenAI
from langchain_community.llms import HuggingFacePipeline
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

/tmp/ipykernel_19196/2329445168.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


In [4]:
os.environ["OPENAI_API_KEY"] = "key"

# LLMs

Integration with many LLM providers
- OpenAI
- Cohere
- AI21
- Huggingface Hub
- Azure OpenAI
- Manifest
- Goose AI
- Writer
- Banana
- Modal
- StochasticAI
- Cerebrium
- Petals
- Forefront AI
- PromptLayer OpenAI
- Anthropic
- DeepInfra
- Self-Hosted Models

In [5]:
gpt3 = ChatOpenAI(model_name="gpt-3.5-turbo")
gpt4_1 = ChatOpenAI(model_name='gpt-4.1')
text = "How to be become an AI expert?"

In [6]:
gpt3.invoke([HumanMessage(content=text)])

AIMessage(content="Becoming an AI expert requires a combination of education, hands-on experience, and continuous learning. Here are some steps you can take to become an AI expert:\n\n1. Education: Obtain a bachelor's degree in computer science, artificial intelligence, data science, or a related field. Consider pursuing a graduate degree or certification in AI if you want to specialize further.\n\n2. Learn programming languages: Become proficient in programming languages commonly used in AI development, such as Python, R, Java, or C++.\n\n3. Gain hands-on experience: Apply your knowledge by working on AI projects, participating in hackathons, or joining AI-related clubs and organizations. Internships or job opportunities in AI-related fields can also provide valuable hands-on experience.\n\n4. Specialize: Choose a specific area within AI to focus on, such as machine learning, natural language processing, computer vision, or robotics.\n\n5. Stay updated: AI is a rapidly evolving field,

In [7]:
gpt4_1.invoke([HumanMessage(content=text)])

AIMessage(content='Becoming an **AI expert** involves a combination of formal education, self-directed learning, practical experience, and ongoing engagement with the rapidly evolving field. Here’s a roadmap to guide you:\n\n---\n\n### **1. Master the Fundamentals**\n\n- **Mathematics:**  \n  - **Linear Algebra** (vectors, matrices, eigenvalues)\n  - **Calculus** (derivatives, gradients, partial derivatives)\n  - **Probability & Statistics** (distributions, Bayes theorem, inference)\n\n- **Programming:**\n  - Strong proficiency in **Python** (most common AI language)\n  - Learn libraries: **NumPy, pandas, matplotlib**\n  \n- **Computer Science Foundations:**\n  - Data structures and algorithms\n  - Software development best practices\n\n---\n\n### **2. Formal Education** (Optional but Helpful)\n\n- **Bachelor’s Degree:** Computer Science, Mathematics, Data Science, or related field\n- **Master’s/PhD:** Specialize in Artificial Intelligence, Machine Learning, or related\n- **Online Cour

# Output Parser

In [8]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1. Define the desired data structure using Pydantic.
class Actor(BaseModel):
  name: str = Field(description="The name of an actor.")
  film_name: str = Field(description="A list of films they have starred in.")

# 2. Create a parser instance from the Pydantic class.
parser = PydanticOutputParser(pydantic_object=Actor)
# 3. Get the auto-generated formatting instructions from the parser.
format_instructions = parser.get_format_instructions()
print("----------- FORMAT INSTRUCTIONS GENERATED BY THE PARSER -----------")
print(format_instructions)
print("-----------------------------------------------------------------")
# 4. Create a prompt template that includes the auto-generated instructions.
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions":format_instructions},
)


----------- FORMAT INSTRUCTIONS GENERATED BY THE PARSER -----------
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "The name of an actor.", "title": "Name", "type": "string"}, "film_name": {"description": "A list of films they have starred in.", "title": "Film Name", "type": "string"}}, "required": ["name", "film_name"]}
```
-----------------------------------------------------------------


In [9]:
# 5. Define the model and build the chain using LCEL.
model = ChatOpenAI(temperature=0, model="gpt-4.1-mini")
chain = prompt | model | parser
# 6. Invoke the chain. The prompt will be sent to the LLM with the formatting guide.
actor_query = "Who is Tom Hanks?"
parsed_output = chain.invoke({"query":actor_query})
print(parsed_output)

name='Tom Hanks' film_name='Forrest Gump'


# Question Answering with external document

There are a lot of document loaders:
File Loader, Directory Loader, Notion, ReadTheDocs, HTML, PDF, PowerPoint, Email, GoogleDrive, Obsidian, Roam, EverNote, YouTube, Hacker News, GitBook, S3 File, S3 Directory, GCS File, GCS Directory, Web Base, IMSDb, AZLyrics, College Confidential, Gutenberg, Airbyte Json, CoNLL-U, iFixit, Notebook, Copypaste, CSV, Facebook Chat, Image, Markdown, SRT, Telegram, URL, Word Document, Blackboard

In [10]:
import requests
url = 'https://s3.amazonaws.com/weclouddata/datasets/genai/langchain/state_of_the_union.txt'
response = requests.get(url)
with open('state_of_the_union.txt','wb') as f:
  f.write(response.content)

print("Document downloaded successfully.")

Document downloaded successfully.


In [11]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("state_of_the_union.txt")
documents = loader.load()
print(f"Loaded {len(documents)} document(s).")
print(f"First 500 characters of the document:\n'{documents[0].page_content[:500]}'")

Loaded 1 document(s).
First 500 characters of the document:
'Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six day'


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)
print(f"Original document has {len(response.content)} words.")
print(f"Document split into {len(texts)} chunks.")
print(f"Content of the first chunk:\n'{texts[0].page_content}'")

Original document has 39027 words.
Document split into 49 chunks.
Content of the first chunk:
'Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their 

In [13]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

print("Embedding model initialized.")


## --- OPTION 2: Hugging Face Embeddings (Local, private, no API key) ---
## Downloads a ~420MB model the first time it's used.
# model_name = "all-mpnet-base-v2"
# embeddings = HuggingFaceEmbeddings(model_name=model_name)


Embedding model initialized.


In [14]:
from langchain_community.vectorstores import FAISS
# 1. Create a FAISS vector store from the text chunks and their embeddings.
print("Creating FAISS vector store...")
vectorstore = FAISS.from_documents(texts, embeddings)
# 2. Create a retriever from the vector store.
retriever = vectorstore.as_retriever()
print("Vector store and retriever are ready.")

Creating FAISS vector store...
Vector store and retriever are ready.


In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Define the LLM we will use for answering
model = ChatOpenAI(model="gpt-4.1-mini")
# 2. Create a prompt template
template = """
Answer the question based only on the following context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
# 3. A helper function to format the retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
# 4. Build the RAG chain using LCEL
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)
# 5. Define our question
query = "What did the president say about Ketanji Brown Jackson?"

In [16]:
# --- Inspection Step: See the retrieved context and final prompt ---

# 5a. First, retrieve the relevant documents for the query
print("----------- RETRIEVING DOCUMENTS -----------")
retrieved_docs = retriever.invoke(query)

# 5b. Format the documents into a single string
formatted_context = format_docs(retrieved_docs)
print("----------- FORMATTED CONTEXT FROM RETRIEVER -----------")
print(formatted_context)

# 5c. See the final prompt that will be sent to the LLM
final_prompt = prompt.invoke({"context": formatted_context, "question": query})
print("\n----------- FINAL PROMPT SENT TO LLM -----------")
# We print the first message content to see the formatted string
print(final_prompt.to_messages()[0].content)

----------- RETRIEVING DOCUMENTS -----------
----------- FORMATTED CONTEXT FROM RETRIEVER -----------
And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence. 

A former top litigator in private practice. A former federal public defender. And from a family of public school educators and police officers. A consensus builder. Since she’s been nominated, she’s received a broad range of support—from the Fraternal Order of Police to former judges appointed by Democrats and Republicans. 

And if we are to advance liberty and justice, we need to secure the Border and fix the immigration system. 

We can do both. At our border, we’ve installed new technology like cutting-edge scanners to better detect drug smuggling.  

We’ve set up joint patrols with Mexico and Guatemala to catch more human traffickers.  

We’re putting in place dedicated immigration judges so

In [17]:
# 6. Invoke the full chain to get the final answer from the LLM
answer = rag_chain.invoke(query)

print("\n\n----------- QUESTION -----------")
print(query)
print("\n----------- FINAL ANSWER -----------")
print(answer)



----------- QUESTION -----------
What did the president say about Ketanji Brown Jackson?

----------- FINAL ANSWER -----------
The president said that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days ago. He described her as one of the nation’s top legal minds who will continue Justice Breyer’s legacy of excellence.


# Keep memory

In [18]:
from langchain_openai import ChatOpenAI
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. Define the LLM
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
# 2. Create a simple store to hold conversation histories for different sessions
store = {}
# 3. Create a function that gets or creates a history object for a given session_id
def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 4. Create the prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly and talkative AI assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{input}")
])
# 5. Build the runnable chain
chain = prompt | llm
# 6. Wrap the chain with memory management
conversation_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history, # Pass the function here
    input_messages_key="input",
    history_messages_key="history",
)

print("Conversational chain with memory is ready.")

Conversational chain with memory is ready.


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [19]:
session_id = "my_first_conversation"

# --- First Turn ---
print("--- Turn 1 ---")
response1 = conversation_with_memory.invoke(
    {"input": "Hi! My name is Alex. Tell me a joke"},
    config={"configurable": {"session_id": session_id}}
)
print(f"AI: {response1.content}")


# --- Second Turn ---
# The chain now remembers the previous turn.
print("\n--- Turn 2 ---")
response2 = conversation_with_memory.invoke(
    {"input": "What's my name?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"AI: {response2.content}")

# --- Third Turn ---
# The chain now remembers the previous turn.
print("\n--- Turn 3 ---")
response3 = conversation_with_memory.invoke(
    {"input": "Tell me a similar joke"},
    config={"configurable": {"session_id": session_id}}
)
print(f"AI: {response3.content}")

# (Optional) You can inspect the memory to see what's stored
print("\n--- Current Memory for session '{}' ---".format(session_id))
print(store[session_id].messages)

--- Turn 1 ---
AI: Hi Alex! Sure, here’s a joke for you:

Why don’t scientists trust atoms?

Because they make up everything! 😄

Want to hear another one?

--- Turn 2 ---
AI: Your name is Alex! Want me to tell you another joke?

--- Turn 3 ---
AI: Sure thing, Alex! Here’s a similar one for you:

Why did the photon check a suitcase at the airport?

Because it was traveling light! 😄

Want to hear more?

--- Current Memory for session 'my_first_conversation' ---
[HumanMessage(content='Hi! My name is Alex. Tell me a joke', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi Alex! Sure, here’s a joke for you:\n\nWhy don’t scientists trust atoms?\n\nBecause they make up everything! 😄\n\nWant to hear another one?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 32, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tok

# Chain

In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Define the model we'll use for both chains
chat = ChatOpenAI(model="gpt-4.1-mini", temperature=0.9)

# Define the standard output parser
parser = StrOutputParser()
# Define the prompt for the first chain (generates a name)
prompt1 = ChatPromptTemplate.from_template(
    "What is a good name for a company that makes {product}?, just show me 1"
)
# Define the prompt for the second chain (generates a catchphrase)
prompt2 = ChatPromptTemplate.from_template(
    "Write a catchphrase for the following company: {company_name}"
)

In [21]:
chain1 = prompt1 | chat | parser

company_name = chain1.invoke({"product":"colorful socks"})
print("----------- OUTPUT OF CHAIN 1 -----------")
print(f"Generated Company Name: {company_name}")

----------- OUTPUT OF CHAIN 1 -----------
Generated Company Name: BrightStep


In [22]:
chain2 = prompt2 | chat | parser

catchphrase = chain2.invoke({"company_name": company_name})

print("----------- OUTPUT OF CHAIN 2 -----------")
print(f"Generated Catchphrase: {catchphrase}")

----------- OUTPUT OF CHAIN 2 -----------
Generated Catchphrase: BrightStep: Illuminating Your Path to Success!


In [23]:
chain2_modified = {"company_name": RunnablePassthrough()} | prompt2 | chat | parser

overall_chain =  chain1 | chain2_modified

final_catchphrase = overall_chain.invoke({"product": "colorful socks"})
print("----------- OUTPUT OF THE COMBINED CHAIN -----------")
print(f"Final Catchphrase: {final_catchphrase}")

----------- OUTPUT OF THE COMBINED CHAIN -----------
Final Catchphrase: Step Into Comfort, Shine With Every Step!


# Agent

This exercise requires google search API key. Here're the steps:
> 1. Go to `https://serpapi.com/`
> 2. Login with your gmail
> 3. Verify your email (and phone sometimes)
> 4. Subscribe to the free plan

In [24]:
!pip install google-search-results

In [25]:
!pip install google-serp-api

ERROR: Could not find a version that satisfies the requirement install (from versions: none)
ERROR: No matching distribution found for install


In [26]:
!pip install langgraph

In [27]:
os.environ["SERPAPI_API_KEY"] = 'key'

In [32]:
!pip install mypy-extensions

In [33]:
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits.load_tools import load_tools
from langgraph.prebuilt import create_react_agent

# 1. Define the LLM the agent will use
llm = ChatOpenAI(model="gpt-4.1-mini",temperature=0)
# 2. Load the tools the agent can use
tools = load_tools(["serpapi", "llm-math"], llm=llm)
agent = create_react_agent(llm,tools)
query = "Who is Leo DiCaprio's girlfriend? What is her current age raised to the 0.43 power?"
result = agent.invoke({"messages": [("user", query)]})
print(result["messages"][-1].content)

/tmp/ipykernel_19196/3808610458.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm,tools)


Leonardo DiCaprio's current girlfriend is Vittoria Ceretti, who is 27 years old. When her age is raised to the power of 0.43, the result is approximately 4.13.
